In [ ]:
# Camera pose estimation from Colmap's rotation and translation matrix

The goal is to estimate the cameras' position (X, Y, Z coordinates) and orientation (pan, tilt, roll) from Colmap's rotation and translation matrix.

In [ ]:
import json
import os

import numpy as np

In [ ]:
data_path = "/data/ROMI/Romi_Alexis/analyse_jo/ahp6_E1/ahp6-1_E1_1/metadata/images"

In [ ]:
def compute_camera_pose(rotation_matrix, translation_vector):
    from scipy.spatial.transform import Rotation as R

    # Compute the camera position in world coordinates
    camera_position = -np.transpose(rotation_matrix) @ translation_vector

    # Extract Euler angles (ZXY) from rotation matrix
    rotation = R.from_matrix(rotation_matrix)
    pan, tilt, roll = rotation.as_euler('zxy', degrees=True)
    pan = pan % 360

    return camera_position, (pan, tilt, roll)

In [ ]:
camera_positions = []
euler_angles = []
rotations = []
for i, filename in enumerate(sorted(os.listdir(data_path))):
    if filename.endswith(".json"):
        with open(os.path.join(data_path, filename)) as f:
            # Load data from the JSON file into a dictionary
            data = json.load(f)
            # Extract rotation matrix from the loaded data for camera pose estimation
            R = np.array(data["colmap_camera"]["rotmat"])
            rotations.append(R)
            # Extract translation vector from the loaded data for camera pose estimation
            t = np.array(data["colmap_camera"]["tvec"])
            # Compute camera position and orientation (pan, tilt, roll) using extracted R and t
            camera_position, euler_angle = compute_camera_pose(R, t)
            #print(f"{i}: ", compute_camera_pose(R, t))
            camera_positions.append(camera_position)
            euler_angles.append(euler_angle)

In [ ]:
np.array(euler_angles)[:, 0]

In [ ]:
import plotly.graph_objects as go


def compute_direction_vector(pan, tilt, roll):
    from scipy.spatial.transform import Rotation as R
    # Create a rotation object from Euler angles
    rotation = R.from_euler('zyx', [pan, tilt, roll], degrees=True)
    # Get rotation matrix
    rot_matrix = rotation.as_matrix()
    # Forward direction in camera's local frame (looking along the negative z-axis originally)
    forward_vector = np.array([0, 0, -1])  # Changed to negative z-axis
    # Rotate forward vector according to rotation matrix
    direction_vector = rot_matrix @ forward_vector
    return direction_vector


def plot_cameras(camera_positions, euler_angles):
    fig = go.Figure()

    for i, (pos, angles) in enumerate(zip(camera_positions, euler_angles)):
        pan, tilt, roll = angles
        direction = compute_direction_vector(pan, tilt, roll)

        # Plot camera position
        fig.add_trace(go.Scatter3d(
            x=[pos[0]], y=[pos[1]], z=[pos[2]],
            mode='markers',
            marker=dict(size=5, color='blue'),
            name=f'Camera {i}'
        ))

        # Plot orientation arrow
        fig.add_trace(go.Cone(
            x=[pos[0]], y=[pos[1]], z=[pos[2]],
            u=[direction[0]], v=[direction[1]], w=[direction[2]],
            sizemode='absolute',
            sizeref=50,
            colorscale='Reds',
            showscale=False,
        ))
    fig.update_layout(width=1000, height=800, showlegend=True)
    fig.show()

In [ ]:
plot_cameras(camera_positions, euler_angles)

In [ ]:
camera_positions

In [ ]:
def plot_cameras(camera_positions, rotations):
    fig = go.Figure()

    for i, (pos, direction) in enumerate(zip(camera_positions, rotations)):
        # Plot camera position
        fig.add_trace(go.Scatter3d(
            x=[pos[0]], y=[pos[1]], z=[pos[2]],
            mode='markers',
            marker=dict(size=5, color='blue'),
            name=f'Camera {i}'
        ))

        # Plot orientation arrow
        fig.add_trace(go.Cone(
            x=[pos[0]], y=[pos[1]], z=[pos[2]],
            u=[direction[0]], v=[direction[1]], w=[direction[2]],
            sizemode='scaled',
            sizeref=50,
            colorscale='Reds',
            showscale=False,
        ))
    fig.update_layout(width=1000, height=800, showlegend=True)
    fig.show()


In [ ]:
plot_cameras(camera_positions, rotations)